# Centerpiece drop-in: add UPVR + O2 to the four-axis stress test (TABLE 0)

Matches the conventions of your existing notebooks (`scan_sota_v3_paired`, `scan_table4_ci_autoattack`):
images kept in **[0,255]**, `pp(x)=(x/255-mean)/std`, CIFAR vs ImageNet normalization, `load_backbone`
with `fc=Linear(2048,nc)` + `['state_dict']`, `mixed_dataset.pkl` caches, and the `auc_ci` bootstrap.

**Two ways to run**
- **Reuse kernel (preferred, zero divergence):** run your existing notebook first so `pp`, `gb`,
  `to224`, `load_backbone`, `feat_hfe`, `feat_gl`, `feat_predl1`, `auc_ci`, `find_mixed` are defined.
  Then run only the cells marked **[NEW]** and **[HARNESS]** below.
- **Standalone:** also run the **[PREAMBLE]** cell, which re-defines those helpers exactly as in your
  cell 1 / cell 2. If your originals differ, prefer the reuse-kernel path.

Output: fills **TABLE 0** for the pixel/frequency detectors {HF-Energy, DCT-HFE, DFT-HFE, UPVR, O2}
with the prediction-compression contrast {GaussianL1, PredL1}, across the four axes. Saved to
`centerpiece_results/`.

Conventions: `#` comments only, ASCII '-' only.


In [ ]:
# ===================== [PREAMBLE] standalone helpers (skip if reusing your kernel) =====================
import os, io as _io, subprocess, pickle
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision.models as models
from torchvision.models import ResNet50_Weights
from torchvision.transforms.functional import gaussian_blur
from PIL import Image as _Image
from sklearn.metrics import roc_auc_score

SEED=42; device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
np.random.seed(SEED); torch.manual_seed(SEED)

def _find(name, maxdepth=8):
    roots=['/home','/root','/workspace',os.path.expanduser('~'),'.','..','../..','.']
    res=[]
    for r in roots:
        if not os.path.exists(r): continue
        try:
            out=subprocess.run(['find',r,'-maxdepth',str(maxdepth),'-type','f','-name',name],
                               capture_output=True,text=True,timeout=20).stdout.strip()
            if out: res+=[p for p in out.split('\n') if p]
        except: pass
    return sorted(set(res))

CIFAR_MEAN=[0.4914,0.4822,0.4465]; CIFAR_STD=[0.2470,0.2435,0.2616]
IMGNET_MEAN=[0.485,0.456,0.406]; IMGNET_STD=[0.229,0.224,0.225]
def make_pp(ds):
    m,s=(CIFAR_MEAN,CIFAR_STD) if 'CIFAR' in ds else (IMGNET_MEAN,IMGNET_STD)
    mean=torch.tensor(m).view(1,3,1,1).to(device); std=torch.tensor(s).view(1,3,1,1).to(device)
    return lambda x:(x/255.0-mean)/std
def load_backbone(ds):
    cfg={'CIFAR-10':('resnet50_cifar10_finetuned.pt',10),'CIFAR-100':('resnet50_cifar100_finetuned.pt',100),
         'SVHN':('resnet50_svhn_finetuned.pt',10),'TinyImageNet':('resnet50_tinyimagenet_finetuned.pt',200)}
    if ds in cfg:
        ck=(_find(cfg[ds][0]) or [None])[0]; m=models.resnet50(weights=None); m.fc=nn.Linear(2048,cfg[ds][1])
        m.load_state_dict(torch.load(ck,map_location=device)['state_dict'])
    else:
        m=models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
    return m.to(device).eval()
def gb(x,s):
    k=int(2*np.ceil(3*s)+1); k=k+1 if k%2==0 else k
    return gaussian_blur(x,kernel_size=k,sigma=s)
def to224(img):
    if img.dim()==3: img=img.unsqueeze(0)
    img=img.float()
    if img.shape[-1]!=224: img=F.interpolate(img,size=(224,224),mode='bicubic',align_corners=False)
    return img.clamp(0,255)
def jpeg(x,q=75):
    a=x.detach().squeeze(0).permute(1,2,0).clamp(0,255).byte().cpu().numpy()
    b=_io.BytesIO(); _Image.fromarray(a).save(b,format='JPEG',quality=int(q)); b.seek(0)
    return torch.from_numpy(np.array(_Image.open(b).convert('RGB'))).float().permute(2,0,1)
def jpeg_batch(b,q=75):
    return torch.stack([jpeg(b[i:i+1]) for i in range(b.shape[0])]).to(b.device)
def median3(x):
    p=F.pad(x,(1,1,1,1),mode='reflect')
    patches=p.unfold(2,3,1).unfold(3,3,1).contiguous().view(*x.shape,9)
    return patches.median(-1).values
def find_mixed():
    out={}
    for p in _find('mixed_dataset.pkl'):
        pl=p.lower()
        if 'cifar10' in pl or 'cifar_10' in pl: out.setdefault('CIFAR-10',p)
        elif 'imagenet' in pl and 'eps8' in pl: out.setdefault('ImageNet',p)
    return out
def auc_ci(neg,pos,B=2000,seed=SEED):
    neg=np.asarray(neg);pos=np.asarray(pos)
    base=roc_auc_score(np.r_[np.zeros(len(neg)),np.ones(len(pos))],np.r_[neg,pos])
    rng=np.random.RandomState(seed); b=[]
    for _ in range(B):
        nb=neg[rng.randint(0,len(neg),len(neg))]; pb=pos[rng.randint(0,len(pos),len(pos))]
        b.append(roc_auc_score(np.r_[np.zeros(len(nb)),np.ones(len(pb))],np.r_[nb,pb]))
    return base,*np.percentile(b,[2.5,97.5])

# existing feature definitions (reconstructed to match your cell 1 / sota cell 2)
def feat_hfe(b):
    return ((b-gb(b,0.5)).abs().flatten(1).mean(1)/255.0).detach().cpu().numpy()
def feat_gl(b,bb,pp,glsig):
    with torch.no_grad():
        p0=F.softmax(bb(pp(b)),1); p1=F.softmax(bb(pp(gb(b,glsig))),1)
    return (p0-p1).abs().sum(1).cpu().numpy()
def feat_predl1(b,bb,pp):
    with torch.no_grad():
        p0=F.softmax(bb(pp(b)),1)
        sq=median3(jpeg_batch(b)).clamp(0,255); p2=F.softmax(bb(pp(sq)),1)
    return (p0-p2).abs().sum(1).cpu().numpy()
print('[PREAMBLE] helpers ready')


In [ ]:
# ===================== [NEW] UPVR, DCT-HFE, DFT-HFE, O2, FGSM =====================
# All operate on b in [0,255], shape [N,3,224,224]. They return raw 1-D statistics;
# the harness standardizes every detector identically via |(f-mu_clean)/sd_clean|,
# which is also what plays the role of O2's per-class calibration here.

def _gray255(b):
    w=torch.tensor([0.299,0.587,0.114],device=b.device).view(1,3,1,1)
    return (b*w).sum(1)  # [N,H,W]

def feat_upvr(b):
    a=b.round().clamp(0,255).to(torch.int32).cpu().numpy()
    N=a.shape[0]; out=np.zeros(N)
    for n in range(N):
        tot=a[n].size
        u=sum(len(np.unique(a[n,c])) for c in range(a.shape[1]))
        out[n]=u/tot
    return out

def feat_dct_hfe(b, B=8):
    from scipy.fft import dctn
    g=_gray255(b).cpu().numpy(); N,H,W=g.shape
    Hc,Wc=(H//B)*B,(W//B)*B; out=np.zeros(N)
    for n in range(N):
        img=g[n,:Hc,:Wc]; acc=0.0; cnt=0
        for i in range(0,Hc,B):
            for j in range(0,Wc,B):
                blk=dctn(img[i:i+B,j:j+B],norm='ortho')
                hf=blk.copy(); hf[:B//2,:B//2]=0.0
                acc+=float(np.sqrt(np.mean(hf**2))); cnt+=1
        out[n]=acc/max(cnt,1)
    return out

def feat_dft_hfe(b, ring_r=0.25):
    g=_gray255(b).cpu().numpy(); N,H,W=g.shape
    yy,xx=np.mgrid[0:H,0:W]; cy,cx=H/2.0,W/2.0
    r=np.sqrt(((yy-cy)/cy)**2+((xx-cx)/cx)**2); ring=(r>=ring_r).astype(np.float32)
    out=np.zeros(N)
    for n in range(N):
        sp=np.abs(np.fft.fftshift(np.fft.fft2(g[n])))
        out[n]=float((sp*ring).sum()/(ring.sum()+1e-8))
    return out

def feat_o2(b, bb, pp, n_noise=16, sigma255=32.0):
    with torch.no_grad():
        base=bb(pp(b)); y=base.argmax(1)
        acc=torch.zeros_like(base)
        for _ in range(n_noise):
            xn=(b+torch.randn_like(b)*sigma255).clamp(0,255); acc+=bb(pp(xn))
        noisy=acc/n_noise
        lo0=base-base.gather(1,y.view(-1,1)); lon=noisy-noisy.gather(1,y.view(-1,1))
        g=(lon-lo0).cpu().numpy()
    for n in range(g.shape[0]): g[n,int(y[n].item())]=-np.inf
    return g.max(1)  # large positive log-odds recovery toward another class -> adversarial-like

def fgsm(x, y, bb, pp, eps255=8.0):
    x=x.clone().to(device).requires_grad_(True)
    loss=F.cross_entropy(bb(pp(x)),y.to(device))
    g,=torch.autograd.grad(loss,x)
    return (x+eps255*g.sign()).clamp(0,255).detach()
print('[NEW] detectors ready')


In [ ]:
# ===================== [HARNESS] axes 1, 2, 4 from mixed_dataset.pkl =====================
N_CLEAN=500; FPRS=[0.01,0.05,0.10]
RESULTS={}

def batched(fn_feat, X, bs=64):
    out=[]
    for i in range(0,len(X),bs):
        out.append(fn_feat(X[i:i+bs].to(device)))
    return np.concatenate(out)

def detector_features(X, bb, pp, glsig):
    # returns dict name -> raw stat array, X is [N,3,224,224] in [0,255] CPU
    return {
        'HF-Energy': batched(lambda b: feat_hfe(b), X),
        'DCT-HFE':   batched(lambda b: feat_dct_hfe(b), X),
        'DFT-HFE':   batched(lambda b: feat_dft_hfe(b), X),
        'UPVR':      batched(lambda b: feat_upvr(b), X),
        'O2':        batched(lambda b: feat_o2(b, bb, pp), X),
        'GaussianL1':batched(lambda b: feat_gl(b, bb, pp, glsig), X),
        'PredL1':    batched(lambda b: feat_predl1(b, bb, pp), X),
    }

def half(n, seed=SEED):
    rng=np.random.RandomState(seed); idx=np.arange(n); rng.shuffle(idx); return idx[:n//2], idx[n//2:]

def tpr_at_fpr(neg, pos, fpr):
    thr=np.quantile(neg, 1-fpr); return float((np.asarray(pos)>=thr).mean())

MX=find_mixed()
for ds, pkl in MX.items():
    bb=load_backbone(ds); pp=make_pp(ds); glsig=0.5 if ds=='CIFAR-10' else 1.0
    mixed=pickle.load(open(pkl,'rb'))
    clean=[to224(im).cpu() for (im,lb,atk) in mixed if atk=='clean']
    adv  =[to224(im).cpu() for (im,lb,atk) in mixed if atk!='clean']
    rng=np.random.RandomState(SEED)
    clean=[clean[i] for i in rng.permutation(len(clean))[:N_CLEAN]]
    Xc=torch.cat(clean,0); Xa=torch.cat(adv,0)
    # hard negatives on the clean test half: matched-budget noise(8) + jpeg(75) + blur(1.0)
    ci,ti=half(len(clean)); te=[clean[i] for i in ti]
    Xte=torch.cat(te,0)
    hn_noise=(Xte+torch.randn_like(Xte)*8.0).clamp(0,255)
    hn_jpeg =jpeg_batch(Xte.to(device)).cpu()
    hn_blur =gb(Xte.to(device),1.0).clamp(0,255).cpu()
    Xhn=torch.cat([hn_noise,hn_jpeg,hn_blur],0)

    fc=detector_features(Xc, bb, pp, glsig)
    fa=detector_features(Xa, bb, pp, glsig)
    fh=detector_features(Xhn, bb, pp, glsig)

    cal_idx, te_idx = ci, ti
    RESULTS[ds]={}
    for name in fc:
        c=fc[name]; mu=c[cal_idx].mean(); sd=c[cal_idx].std()+1e-8
        anom=lambda v: np.abs((v-mu)/sd)
        neg_p=anom(c[te_idx]); pos=anom(fa[name]); neg_hn=np.concatenate([anom(c[te_idx]), anom(fh[name])])
        a_p,lo_p,hi_p=auc_ci(neg_p,pos); a_h,lo_h,hi_h=auc_ci(neg_hn,pos)
        RESULTS[ds][name]={
            'pristine_auroc':round(float(a_p),4),'pristine_ci':[round(float(lo_p),4),round(float(hi_p),4)],
            'hardneg_auroc':round(float(a_h),4),'hardneg_ci':[round(float(lo_h),4),round(float(hi_h),4)],
            'delta_auc':round(float(a_h-a_p),4),
            'tpr_pristine':{f'{int(f*100)}%':round(tpr_at_fpr(neg_p,pos,f),4) for f in FPRS},
            'tpr_operational':{f'{int(f*100)}%':round(tpr_at_fpr(neg_hn,pos,f),4) for f in FPRS},
        }
    print(f'[{ds}] axes 1/2/4 done  (clean={len(clean)} adv={Xa.shape[0]} hn={Xhn.shape[0]})')
import pandas as pd
pd.DataFrame({d:{n:RESULTS[d][n]['delta_auc'] for n in RESULTS[d]} for d in RESULTS})


In [ ]:
# ===================== [HARNESS] axis 3 controlled resolution (ImageNet only) =====================
RES_RATIOS=[1.0,1.4,2.0,3.5,7.0]; RES_N=200
AXIS3={}
if 'ImageNet' in MX:
    bb=load_backbone('ImageNet'); pp=make_pp('ImageNet'); glsig=1.0
    mixed=pickle.load(open(MX['ImageNet'],'rb'))
    clean=[to224(im).cpu() for (im,lb,atk) in mixed if atk=='clean'][:RES_N]
    Xc0=torch.cat(clean,0)
    with torch.no_grad():
        Y0=torch.cat([bb(pp(Xc0[i:i+64].to(device))).argmax(1).cpu() for i in range(0,len(Xc0),64)])
    for ratio in RES_RATIOS:
        small=max(32,int(round(224/ratio)))
        ds_=F.interpolate(Xc0,size=small,mode='bicubic',align_corners=False).clamp(0,255)
        us =F.interpolate(ds_,size=224,mode='bicubic',align_corners=False).clamp(0,255)
        adv=[]
        for i in range(0,len(us),64):
            adv.append(fgsm(us[i:i+64].to(device),Y0[i:i+64],bb,pp).cpu())
        Xa=torch.cat(adv,0)
        fc=detector_features(us, bb, pp, glsig); fa=detector_features(Xa, bb, pp, glsig)
        AXIS3[ratio]={}
        for name in fc:
            c=fc[name]; mu=c.mean(); sd=c.std()+1e-8
            an=lambda v: np.abs((v-mu)/sd)
            from sklearn.metrics import roc_auc_score as _auc
            y=np.r_[np.zeros(len(c)),np.ones(len(fa[name]))]
            AXIS3[ratio][name]=round(float(_auc(y,np.r_[an(c),an(fa[name])])),4)
        print(f'  ratio x{ratio}: native-HF-energy proxy done')
    print('[axis 3] controlled resolution done')
else:
    print('[axis 3] no ImageNet mixed cache found; skip')
AXIS3


In [ ]:
# ===================== assemble TABLE 0 and save =====================
import json, pandas as pd
OUT='./centerpiece_results'; os.makedirs(OUT,exist_ok=True)
DETS=['HF-Energy','DCT-HFE','DFT-HFE','UPVR','O2','GaussianL1','PredL1']
PIXEL_FREQ=['HF-Energy','DCT-HFE','DFT-HFE','UPVR','O2']

def verdict(ds,name):
    r=RESULTS[ds][name]; collapses=r['delta_auc']<=-0.1; op0=r['tpr_operational'].get('5%',1.0)<=0.05
    if name in ('GaussianL1','PredL1'): return 'adversarial (C&W/PGD)'
    return 'perturbation detector' if (collapses or op0) else 'holds (inspect axes)'

ds_main='CIFAR-10' if 'CIFAR-10' in RESULTS else (list(RESULTS)[0] if RESULTS else None)
rows=[]
for name in DETS:
    if ds_main is None: break
    r=RESULTS[ds_main][name]
    rows.append({
        'detector':name,
        '(1) pristine AUROC':r['pristine_auroc'],
        '(2) +hard-neg AUROC':r['hardneg_auroc'],
        '(2) delta-AUC':r['delta_auc'],
        '(3) native AUROC (IN)':AXIS3.get(1.0,{}).get(name,'n/a'),
        '(4) oper. TPR@5%FPR':r['tpr_operational'].get('5%'),
        'verdict':verdict(ds_main,name),
    })
df0=pd.DataFrame(rows)
json.dump({'RESULTS':RESULTS,'AXIS3':AXIS3}, open(os.path.join(OUT,'centerpiece_results.json'),'w'), indent=2)
df0.to_csv(os.path.join(OUT,'centerpiece_table0.csv'),index=False)
print('saved to',OUT); df0
